In [7]:
!pip install numpy torch torchsummary scikit-learn pandas plotly scipy mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 752.6/752.6 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 14.6 MB/s eta 0:00:00


In [8]:
# improtar bibliotecas

# Deep Learning / ML
import numpy as np
from torch import nn
import torch
from torch.utils.data import DataLoader, Dataset
from torchsummary import summary
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# EDA
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import chi2_contingency

# MLFlow (MlOps)
import mlflow
import mlflow.pytorch

## Carga dos Dados

In [9]:
# Carregar o dataset
df_passagens = pd.read_csv('data/dataset_passagens.csv')

### Exploração Inicial dos Dados

In [10]:
df_passagens.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18560 entries, 0 to 18559
Data columns (total 12 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Localizador                         18560 non-null  object 
 1   Cia Aérea                           18560 non-null  object 
 2   Origem                              18560 non-null  object 
 3   Destino                             18560 non-null  object 
 4   Classe da Passagem                  18560 non-null  int64  
 5   Qtde de Paradas                     18560 non-null  int64  
 6   Tipo de Destino                     18560 non-null  object 
 7   Estação do Ano                      18560 non-null  object 
 8   Preço Atual da Passagem             18560 non-null  float64
 9   Preço da Passagem na última semana  18560 non-null  float64
 10  Preço da Passagem no último mês     18560 non-null  float64
 11  Decisão do Cliente                  18560

In [11]:
# Visualizar primeiros registros
df_passagens.head(10)

,Localizador,Cia Aérea,Origem,Destino,Classe da Passagem,Qtde de Paradas,Tipo de Destino,Estação do Ano,Preço Atual da Passagem,Preço da Passagem na última semana,Preço da Passagem no último mês,Decisão do Cliente
0,EECE52,QATAR,DUB,EWR,3,0,romântico,inverno,5571.43,5444.73,4718.87,Não comprou
1,0GQ2XB,LATAM,PVG,BOM,4,1,romântico,outono,9914.27,8105.21,7951.10,Não comprou
2,9LC06Q,AMERICAN,MAD,DEL,3,1,negócios,primavera,5896.35,6897.04,7423.17,Comprou
3,8CXT02,QATAR,ZRH,DOH,1,1,aventura,primavera,893.75,1069.73,1008.56,Comprou
4,LKZEF5,AMERICAN,MEL,LGW,3,2,família,primavera,5329.99,5529.97,6097.89,Preferiu aguardar e comprou depois
5,BQANJQ,EMIRATES,JNB,JFK,1,2,histórico,inverno,1238.14,1076.75,927.16,Não comprou
6,9HNM6Q,UNITED,PVG,MEL,3,0,aventura,primavera,6274.16,6811.84,7843.19,Preferiu aguardar e comprou depois
7,ZNZ66K,AZUL,MXP,BOM,1,2,família,primavera,1049.86,1126.21,918.33,Comprou
8,J7YEQA,LATAM,LAX,PEK,2,1,religioso,primavera,3792.37,4306.77,3994.70,Comprou
9,GNCMKO,QATAR,LAS,AKL,1,1,negócios,verão,2048.14,2401.53,2864.69,Preferiu aguardar e comprou depois


In [13]:
# Estatísticas descritivas do dataset
df_passagens.describe()

,Classe da Passagem,Qtde de Paradas,Preço Atual da Passagem,Preço da Passagem na última semana,Preço da Passagem no último mês
count,18560.000000,18560.000000,18560.000000,18560.000000,18560.000000
mean,2.498330,0.989009,5541.536206,5546.682030,5552.621525
std,1.119315,0.814484,4014.934804,4095.224134,4182.750004
min,1.000000,0.000000,681.260000,565.870000,471.670000
25%,1.000000,0.000000,2027.530000,2020.912500,2025.880000
50%,2.000000,1.000000,4649.300000,4591.210000,4530.080000
75%,4.000000,2.000000,7984.667500,7985.670000,7936.027500
max,4.000000,2.000000,20465.380000,23796.420000,27072.160000


In [17]:
# Motrar valores únicos de cada coluna categóricas
for col in df_passagens.select_dtypes(include=['object']):
      print(f'{col}: {df_passagens[col].unique()}')

Localizador: ['EECE52' '0GQ2XB' '9LC06Q' ... 'U96ERS' 'GEE2KZ' 'H1JM1M']
Cia Aérea: ['QATAR' 'LATAM' 'AMERICAN' 'EMIRATES' 'UNITED' 'AZUL' 'KLM' 'DELTA' 'GOL'
 'AIR FRANCE']
Origem: ['DUB' 'PVG' 'MAD' 'ZRH' 'MEL' 'JNB' 'MXP' 'LAX' 'LAS' 'CDG' 'HND' 'PHX'
 'MIA' 'SFO' 'DFW' 'SEA' 'BRU' 'YYZ' 'AMS' 'BCN' 'LGW' 'VIE' 'YVR' 'EWR'
 'FRA' 'NRT' 'ICN' 'LHR' 'IAH' 'HKG' 'FCO' 'LIS' 'LGA' 'DEN' 'DOH' 'DXB'
 'CLT' 'ORD' 'BOS' 'BOM' 'AKL' 'MUC' 'ATL' 'CPT' 'DEL' 'JFK' 'SIN' 'PEK'
 'ATH' 'SYD']
Destino: ['EWR' 'BOM' 'DEL' 'DOH' 'LGW' 'JFK' 'MEL' 'PEK' 'AKL' 'CDG' 'DXB' 'FCO'
 'AMS' 'LAX' 'LGA' 'SIN' 'LHR' 'YYZ' 'HND' 'DFW' 'ORD' 'MAD' 'CPT' 'SYD'
 'ATL' 'MUC' 'ICN' 'PHX' 'ZRH' 'IAH' 'MIA' 'DEN' 'VIE' 'BCN' 'DUB' 'BOS'
 'ATH' 'SEA' 'FRA' 'MXP' 'YVR' 'NRT' 'PVG' 'SFO' 'JNB' 'LAS' 'LIS' 'BRU'
 'HKG' 'CLT']
Tipo de Destino: ['romântico' 'negócios' 'aventura' 'família' 'histórico' 'religioso']
Estação do Ano: ['inverno' 'outono' 'primavera' 'verão']
Decisão do Cliente: ['Não comprou' 'Comprou' 'Preferi

### Dominio - Classe da Passagem

1 - Econômica
2 - Econômica Premium
3 - Executivo
4 - Primeira Classe

## Preparação de Dados para EDA

In [18]:
# Remover colunas única Localizador
df_passagens.drop(columns=['Localizador'], inplace=True)

In [20]:
# Criar uma lista de variáveis categóricas
categorical_features = df_passagens.select_dtypes(include=['object']).columns.tolist()
# Incluir variável categorica
categorical_features.append("Classe da Passagem")
categorical_features.remove("Decisão do Cliente")
categorical_features

['Cia Aérea',
 'Origem',
 'Destino',
 'Tipo de Destino',
 'Estação do Ano',
 'Classe da Passagem']

In [23]:
# Criar lista de variáveis numéricas
numerical_features = df_passagens.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_features.remove("Classe da Passagem")
numerical_features

['Qtde de Paradas',
 'Preço Atual da Passagem',
 'Preço da Passagem na última semana',
 'Preço da Passagem no último mês']

In [24]:
# Target
target = ["Decisão do Cliente"]